# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using the `mlcroissant` library.

### Dataset Source

The dataset is described using a Croissant schema, which formalizes record sets, fields, and their structure. See [Croissant schema documentation](https://mlcommons.org/croissant/).

> **Schema URL:**  
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is already a DatasetMetadata object
print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('\nPublished:', getattr(metadata, 'datePublished', None))
print('Version:', getattr(metadata, 'version', None))
print('License:', getattr(metadata, 'license', None))

## 2. Data Overview

Review available record sets, their fields, and corresponding `@id` values.

In Croissant, record sets organize dataset tables or main data entities. Each field and column is referenced by its `@id` for clarity. We'll use `dataset.record_sets` to explore the dataset contents. 

In [ ]:
print('Available Record Sets:')
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- Name: {rs.name} | @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields for each record set
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (id: {f.id}, dataType: {getattr(f, 'data_type', None)})")
    print('')

## 3. Data Extraction

Load records from the main record sets into DataFrames using their `@id` references.

> **Note:** Most datasets have a primary record set for survey data, and possibly others for dictionary or metadata tables.


In [ ]:
# Prepare to extract all record sets
dataframes = {}
for rs in dataset.record_sets:
    print(f"Loading record set: {rs.id} ({rs.name})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Columns for {rs.name} ({rs.id}):\n    {df.columns.tolist()}\n  Example rows: \n{df.head(2)}\n")

# Select the main record set for further analysis (e.g., the one with GAD-7, PHQ-9, PSQ)

# For this dataset, likely the main survey data is in the first record set:
record_set_id = dataset.record_sets[0].id
print(f'Will use record set id: {record_set_id}')

## 4. Exploratory Data Analysis (EDA)

Let us process and analyze filtered records from the main record set. We'll select a numeric field (e.g., total score for GAD-7 or PHQ-9), reference it with its `@id`, normalize, and group data.

First, we list all numeric fields to choose one for analysis.

In [ ]:
# Identify available numeric fields in the main record set
main_rs = [rs for rs in dataset.record_sets if rs.id == record_set_id][0]
numeric_fields = [f for f in main_rs.fields if getattr(f, 'data_type', '').lower() in ['integer', 'float', 'number']]
print('Numeric fields in record set:', [f"{f.name} (@id: {f.id})" for f in numeric_fields])

# For demonstration, select the first numeric field (adjust as needed)
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    print(f\nSelected numeric field for EDA: {numeric_fields[0].name} (@id: {numeric_field_id})")
else:
    raise RuntimeError("No numeric fields found in main record set!")

df = dataframes[record_set_id]

# Display value counts to help pick a grouping field
print("\nExample value counts for all columns (first 3):")
for c in df.columns[:3]:
    print(f"- {c}")
    print(df[c].value_counts().head(), end="\n\n")

In [ ]:
# Filter records for numeric_field > threshold (adjust threshold as relevant)
threshold = 5  # Example value; choose based on the scale reported in Data Overview
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} records")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a common field, e.g., 'age', 'gender' (referenced by @id)
# Find a suitable field for grouping
group_fields = [f.id for f in main_rs.fields if 'age' in f.id.lower() or 'gender' in f.id.lower() or 'group' in f.id.lower()]
if group_fields:
    group_field = group_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    print("No obvious demographic group field for grouping.")

## 5. Visualization
Visualize the normalized scores and distributions for the chosen field. Grouped bar or violin plots can reveal differences by demographic field (e.g., gender or age group). Plotting will only work if suitable data is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=12, color="skyblue")
plt.title(f"Distribution of {numeric_field_id} (Filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.show()

# If group field selected, show boxplot by group
if group_fields:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and described the FAIR² Mental Health Survey data using `mlcroissant`.
- Explored record sets, their fields, and accessed dataset structure by `@id`.
- Loaded main records data and conducted simple exploratory data analysis: filtering by score, normalization, grouping, and visualization.

Further analyses could include correlating symptoms with demographic features or exploring cross-sections for at-risk groups. For complete details on the FAIR² Croissant schema design, see the [original metadata](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).